# Landing → Bronze

## Objetivo

Receber os arquivos brutos na landing zone e gravar na camada Bronze **sem alteração de conteúdo ou estrutura**, acrescentando apenas a coluna técnica `ingestion_datetime`. O notebook também ingere a cotação do dólar a partir da API PTAX do Banco Central.

## Decisão de arquitetura: um único notebook para ambiente, landing e Bronze

Este notebook reúne três etapas que poderiam ser notebooks separados:

1. **Organização do ambiente:** catalog, schemas (`bronze`, `silver`, `gold`) e volume `Landing`.
2. **Landing:** validação da presença dos 5 arquivos de origem.
3. **Ingestão Bronze:** leitura dos CSVs e da API e gravação das tabelas Delta.

Motivos:

- O Job possui uma task por camada (`to_Bronze`, `to_Silver`, `to_Gold`). Manter ambiente e landing dentro da task `to_Bronze` evita uma task adicional só para preparar a infraestrutura.
- Todos os comandos de criação usam `IF NOT EXISTS`, portanto são idempotentes: o pipeline roda do zero em um workspace vazio e não falha nas execuções seguintes.
- A ingestão só pode começar depois que o ambiente existe e os arquivos foram validados. No mesmo notebook essa ordem é garantida pela própria sequência das células.

## Estrutura do notebook

1. Configuração
2. Organização do ambiente
3. Validação da landing zone
4. Leitura dos arquivos CSV
5. Gravação das tabelas Bronze
6. Cotação do dólar (API PTAX)

## 1. Configuração

Os nomes do catalog, dos schemas e do caminho da landing zone ficam centralizados em variáveis usadas por todas as células, o que permite alterar o ambiente em um único ponto.

Cada schema representa o "banco de dados" de uma camada da arquitetura medalhão: `bronze`, `silver` e `gold`.

In [0]:
catalog = "cineData_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: cineData_analytics
bronze_schema: cineData_analytics.bronze
silver_schema: cineData_analytics.silver
gold_schema: cineData_analytics.gold
landing_path: /Volumes/cineData_analytics/bronze/Landing


## 2. Organização do ambiente

Cria o catalog, os três schemas e o Volume `Landing`, que armazena os CSVs de origem dentro do Unity Catalog, mantendo arquivos e tabelas sob a mesma governança.

Decisão: os schemas `silver` e `gold` são criados aqui, e não nos notebooks das próprias camadas, para que toda a estrutura do projeto exista desde a primeira task do Job.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catalog, schemas e volume prontos.")

Catalog, schemas e volume prontos.


## 3. Validação da landing zone

Confere se os 5 arquivos esperados estão no Volume antes de iniciar a ingestão.

Decisão: se algum arquivo estiver ausente, a célula lança `FileNotFoundError`. A task falha de forma explícita (fail-fast) em vez de gravar uma Bronze incompleta que só seria percebida nas camadas seguintes.

Observação: o arquivo de informações dos filmes é `movies_info_TMDB_IMDB.csv`. O enunciado cita `movies_info_IMDB_TMDB.csv`, mas o código usa o nome real do arquivo recebido.

In [0]:
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
    #interrompe a execução caso algum arquivo esteja ausente
    raise FileNotFoundError("Arquivos ausentes na landing zone. Faça o upload antes de continuar.")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos esperados estão na landing zone:
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)


## 4. Leitura dos arquivos CSV

Os arquivos são lidos com `header=True` e **sem `inferSchema`**, portanto todas as colunas chegam como texto.

Decisão: a Bronze deve espelhar a origem. A tipagem e a limpeza são responsabilidade da Silver. A inferência de tipos poderia converter ou perder valores sujos que precisam ser tratados com regras explícitas (por exemplo `Unknown`, `10.0K` ou datas em formatos diferentes).

In [0]:
from pyspark.sql.functions import current_timestamp

path_credits_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

df_credits_tags_raw = spark.read.csv(path_credits_tags, header=True)
df_movies_financials_raw = spark.read.csv(path_movies_financials, header=True)
df_movies_info_raw = spark.read.csv(path_movies_info, header=True)
df_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True)
df_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True)

## 5. Gravação das tabelas Bronze

Cada CSV vira uma tabela Delta (`tb_movies_info`, `tb_movies_financials`, `tb_movies_metrics`, `tb_credits_and_tags` e `tb_movies_reviews`) com a coluna `ingestion_datetime`, que recebe o timestamp do momento da gravação.

Decisões:

- **Modo `append`**, conforme o enunciado: cada execução acrescenta uma nova versão dos dados e o histórico das cargas é preservado.
- Duplicatas entre execuções são esperadas na Bronze. A Silver as resolve mantendo, por chave, o registro de maior `ingestion_datetime`.
- Nenhuma coluna é removida, renomeada ou convertida nesta camada.

In [0]:
df_movies_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_info")

df_movies_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_financials")

df_movies_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_metrics")

df_credits_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags")

df_movies_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_reviews")

display(spark.table(f"{catalog}.bronze.tb_movies_info").limit(5))
display(spark.table(f"{catalog}.bronze.tb_movies_financials").limit(5))
display(spark.table(f"{catalog}.bronze.tb_movies_metrics").limit(5))
display(spark.table(f"{catalog}.bronze.tb_credits_and_tags").limit(5))
display(spark.table(f"{catalog}.bronze.tb_movies_reviews").limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-20T20:43:08.180Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-20T20:43:08.180Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-20T20:43:08.180Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-20T20:43:08.180Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-20T20:43:08.180Z


id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-09-20T20:43:12.577Z
299536,300000000,2052415039,2026-09-20T20:43:12.577Z
299534,356000000,2800000000,2026-09-20T20:43:12.577Z
475557,55000000,1074458282,2026-09-20T20:43:12.577Z
271110,250000000,Não Informado,2026-09-20T20:43:12.577Z


id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-09-20T20:43:15.375Z
299536,"154,34",8.255,27713,8.4,1406782,2026-09-20T20:43:15.375Z
299534,91.756,8.263,23857,8.4,1484150,2026-09-20T20:43:15.375Z
475557,"54,522",8.168,23425,8.3,1723035,2026-09-20T20:43:15.375Z
271110,70.741,7.4,21541,7.8,947222,2026-09-20T20:43:15.375Z


id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-09-20T20:43:18.403Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-09-20T20:43:18.403Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-09-20T20:43:18.403Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-09-20T20:43:18.403Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-09-20T20:43:18.403Z


id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-09-20T20:43:21.705Z
637007,Lucas Reis 602,3.9,null,2026-09-20T20:43:21.705Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-09-20T20:43:21.705Z
413036,Gabriela Monteiro 401,7.5,null,2026-09-20T20:43:21.705Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-09-20T20:43:21.705Z


## 6. Cotação do dólar (API)

Consulta o endpoint `CotacaoDolarPeriodo` do Banco Central, seleciona `dataHoraCotacao` e `cotacaoCompra` e grava o resultado em `bronze.tb_cotacao_dolar`.

Decisões:

- **Widgets** `data_inicio` e `data_fim` no formato `MM-DD-AAAA`, exigido pela API. Os valores são validados com `strptime` para que uma data inválida falhe com uma mensagem clara.
- **Janela padrão de `01-01-2015` até a data de execução.** O enunciado sugere os últimos 7 dias, mas a conversão para BRL usa a cotação da data de lançamento de cada filme e a base tem filmes lançados desde 2016, então é necessário ter histórico. Para uma rotina incremental basta passar uma janela menor como parâmetro do Job.
- **Falha explícita:** a chamada usa `timeout`, `raise_for_status()` e lança erro se a API retornar vazio, para que o Workflow não siga adiante com uma cotação ausente.
- **Schema explícito** (`STRING` e `DOUBLE`), sem depender da inferência sobre os dicionários do JSON.
- **Modo `append`:** cada execução regrava a janela consultada. A sobreposição entre execuções é resolvida na Silver, que mantém uma única cotação por dia.

In [0]:
from datetime import datetime
import requests
from pyspark.sql.functions import current_timestamp

data_atual = datetime.today()

data_fim_default = data_atual.strftime("%m-%d-%Y")
data_inicio_default = "01-01-2015" # Buscar o histórico de cotações

dbutils.widgets.text("data_inicio", data_inicio_default, "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_default, "Data Fim (MM-DD-AAAA)")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

#valida o formato das datas informadas nos widgets MM-DD-AAAA
datetime.strptime(data_inicio_formatada, "%m-%d-%Y")
datetime.strptime(data_fim_formatada, "%m-%d-%Y")

url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

response = requests.get(url, timeout=30)

#interrompe a execução em caso de erro na API para que a task do workflow falhe de forma explicita
response.raise_for_status()

registros = response.json().get("value", [])

#interrompe a execução caso a API não retorne cotações no período informado
if not registros:
    raise ValueError("A API não retornou cotações para o período informado.")

#schema explícito para não depender da inferência dos dicts do json
df_cotacao = spark.createDataFrame(
    [(r["dataHoraCotacao"], r["cotacaoCompra"]) for r in registros],
    "dataHoraCotacao STRING, cotacaoCompra DOUBLE"
)

(
    df_cotacao
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")
)
print(f"Sucesso! {len(registros)} registros salvos em bronze.tb_cotacao_dolar.")

display(df_cotacao.limit(25))

Sucesso! 2940 registros salvos em bronze.tb_cotacao_dolar.


dataHoraCotacao,cotacaoCompra
2015-01-02 13:09:00.008,2.6923
2015-01-05 13:05:39.045,2.7101
2015-01-06 13:02:40.508,2.7016
2015-01-07 13:03:26.355,2.6801
2015-01-08 13:14:50.913,2.6913
2015-01-09 13:05:35.298,2.6577
2015-01-12 13:08:28.69,2.6569
2015-01-13 13:03:36.247,2.6479
2015-01-14 13:11:33.125,2.6216
2015-01-15 13:09:30.875,2.6116
